# Lab 9: Πολυπρακτορικά Συστήματα με CrewAI

## Σκοπός

Σε αυτό το εργαστήριο θα **κατασκευάσουμε ένα πολυπρακτορικό σύστημα** που προσομοιώνει μια μικρή αναλυτική ομάδα επενδύσεων. Θα δούμε πρακτικά:

- Πώς ορίζονται **AI Agents** με ρόλους, στόχους και ιστορικό (backstory)
- Πώς αναθέτουμε **Tasks** σε συγκεκριμένους πράκτορες
- Πώς μια **Crew** συντονίζει τους πράκτορες και παράγει ένα τελικό αποτέλεσμα
- Πώς η **αλυσίδα επικοινωνίας** μεταξύ πρακτόρων παράγει έξοδο που κανένας μεμονωμένος agent δεν θα παρήγαγε μόνος του

## Σενάριο

Φανταστείτε ότι εργάζεστε σε ένα μικρό venture capital fund. Ο manager σας θέλει μια **γρήγορη αναφορά** για μια εταιρεία πριν από μια επενδυτική συνάντηση. Δεν υπάρχει χρόνος να γίνει χειροκίνητη έρευνα — αναθέτουμε τη δουλειά σε μια **AI Crew**:

| Agent | Ρόλος | Αρμοδιότητα |
|---|---|---|
| **Researcher** | Senior Market Analyst | Συλλέγει πληροφορίες για την εταιρεία |
| **Analyst** | Investment Advisor | Γράφει την τελική αναφορά υπέρ/κατά |

---

## ⚙️ Προαπαιτούμενο: Python 3.13 & Virtual Environment

> ⚠️ Το notebook απαιτεί **Python 3.13**. Η βιβλιοθήκη `tiktoken` (εξάρτηση του CrewAI) δεν υποστηρίζει Python 3.14+.

Αν δεν έχετε ήδη το `lab9venv`, δημιουργήστε νέο περιβάλλον με τα παρακάτω βήματα (εκτελέστε σε terminal, **όχι** μέσα στο notebook):

```bash
# 1. Δημιουργία virtual environment με Python 3.13
python3.13 -m venv lab9venv

# 2. Ενεργοποίηση (Linux/macOS)
source lab9venv/bin/activate

# 3. Εγκατάσταση όλων των βιβλιοθηκών από το αρχείο απαιτήσεων
pip install -r lab9-requirements.txt
```

Στη συνέχεια επιλέξτε το `lab9venv` kernel στο Jupyter/VS Code:
**Kernel → Select Kernel → lab9venv (Python 3.13)**

---

## 🔑 Προαπαιτούμενο: Δωρεάν API Key από το Groq

Χρησιμοποιούμε το **Groq** ως LLM provider — είναι **δωρεάν** και δεν χρειάζεται πιστωτική κάρτα.

1. Πηγαίνετε στο [console.groq.com](https://console.groq.com)
2. Δημιουργήστε λογαριασμό (με Google ή email)
3. Πηγαίνετε στο **API Keys** → **Create API Key**
4. Αντιγράψτε το key (ξεκινάει με `gsk_...`)
5. Επικολλήστε το στο κελί παρακάτω

## Εργαλεία & Βιβλιοθήκες

Σε αυτό το εργαστήριο χρησιμοποιούμε τα παρακάτω εργαλεία:

### CrewAI — Βασικά Components

| Component | Τι είναι | Αναλογία |
|---|---|---|
| `LLM` | Σύνδεση με το γλωσσικό μοντέλο (π.χ. Llama 3.3 μέσω Groq) | Ο «εγκέφαλος» κάθε agent |
| `Agent` | Αυτόνομος πράκτορας με ρόλο, στόχο και προσωπικότητα | Ένας υπάλληλος με τίτλο εργασίας |
| `Task` | Συγκεκριμένη εντολή εργασίας που ανατίθεται σε agent | Ένα task/ticket στο Jira |
| `Crew` | Ενορχηστρωτής που συντονίζει agents και tasks | Ο project manager της ομάδας |
| `Process` | Τρόπος εκτέλεσης: `sequential` (αλυσίδα) ή `hierarchical` (ιεραρχία) | Η δομή της ομάδας |

### Εξωτερικές Βιβλιοθήκες

| Βιβλιοθήκη | Τι κάνει |
|---|---|
| `crewai` | Το κεντρικό framework για δημιουργία πολυπρακτορικών συστημάτων |
| `crewai-tools` | Έτοιμα εργαλεία για τους agents (αναζήτηση web, ανάγνωση αρχείων κ.λπ.) |
| `litellm` | Ενιαίο interface για όλους τους LLM providers (Groq, OpenAI, Anthropic κ.λπ.) |
| `groq` (API) | Δωρεάν, υψηλής ταχύτητας inference για open-source μοντέλα (Llama, Gemma) |

### Τι είναι το Groq;

Το **Groq** είναι μια εταιρεία που κατασκεύασε ειδικό hardware — τον **LPU (Language Processing Unit)** — βελτιστοποιημένο αποκλειστικά για εκτέλεση γλωσσικών μοντέλων. Σε αντίθεση με GPU που είναι general-purpose, το LPU εκτελεί LLM inference με **πολύ χαμηλή καθυστέρηση** και **σταθερή ταχύτητα**.

**Γιατί το χρησιμοποιούμε εδώ;**

- Παρέχει **δωρεάν API** με γενναιόδωρα όρια (free tier)
- Δεν απαιτεί πιστωτική κάρτα για εγγραφή
- Τρέχει open-source μοντέλα (Meta Llama, Google Gemma) χωρίς τοπική εγκατάσταση
- Η ταχύτητά του (~500 tokens/sec) κάνει τις πολυπρακτορικές αλυσίδες πρακτικές στο εργαστήριο

**Αρχιτεκτονική σύνδεσης:**

```
Κώδικας Python (CrewAI)
        ↓  HTTPS request
  api.groq.com  (LPU cloud)
        ↓  token streaming
   Απάντηση του LLM
```

> 💡 **Εναλλακτικοί providers** που δουλεύουν με την ίδια λογική: OpenAI, Anthropic (Claude), Google Gemini, ή τοπικά μοντέλα μέσω Ollama. Αρκεί να αλλάξετε το `model=` στην κλάση `LLM`.

### Ροή Δεδομένων

![CrewAI Data Flow](../img/lec9/lab9_crewai_flow.png)

In [9]:
# Εγκατάσταση βιβλιοθηκών
# Σημείωση: χρησιμοποιεί Python 3.13 (lab9venv) — το tiktoken δεν υποστηρίζει Python 3.14
# Αν τρέχετε για πρώτη φορά σε νέο περιβάλλον, αποσχολιάστε την παρακάτω γραμμή:
%pip install -q crewai crewai-tools

# Επαλήθευση εγκατάστασης
import crewai, sys
print(f"✅ crewai {crewai.__version__} είναι εγκατεστημένο (Python {sys.version.split()[0]})")

Note: you may need to restart the kernel to use updated packages.
✅ crewai 1.14.4 είναι εγκατεστημένο (Python 3.13.12)


## Βήμα 0: Ρύθμιση API Key

Για να επικοινωνούν οι agents με το LLM, χρειαζόμαστε ένα **API Key** από τον provider μας (εδώ: Groq).

Το key αποθηκεύεται ως **μεταβλητή περιβάλλοντος** (`GROQ_API_KEY`) ώστε να μην εμφανίζεται στον κώδικα. Η βιβλιοθήκη CrewAI διαβάζει αυτόματα αυτές τις μεταβλητές κατά την αρχικοποίηση του LLM.

> ⚠️ Μην μοιράζεστε ποτέ το API key σας σε δημόσιο αποθετήριο (GitHub κ.λπ.).

In [5]:
import os

# ✏️ ΒΑΛΤΕ ΕΔΩ ΤΟ GROQ API KEY ΣΑΣ
os.environ["GROQ_API_KEY"] = "gsk_ΧΧΧΧΧΧ"

# Επαλήθευση ότι το key έχει οριστεί
key = os.environ.get("GROQ_API_KEY", "")
if key.startswith("gsk_") and len(key) > 20:
    print("✅ Groq API Key ορίστηκε επιτυχώς.")
else:
    print("❌ Το API Key δεν φαίνεται σωστό. Ελέγξτε ότι ξεκινά με gsk_")

✅ Groq API Key ορίστηκε επιτυχώς.


## Βήμα 1: Ορισμός του LLM

Το CrewAI χρειάζεται έναν **"εγκέφαλο"** για κάθε agent — εδώ χρησιμοποιούμε το **Llama 3.3 70B** μέσω Groq.

Η κλάση `LLM` του CrewAI επικοινωνεί με τον provider (Groq) χρησιμοποιώντας το πρωτόκολλο LiteLLM. Το format του model name είναι: `"provider/model-name"`.


In [1]:
from crewai import LLM

# Ορίζουμε το LLM που θα χρησιμοποιηθεί από όλους τους agents
# Διαθέσιμα δωρεάν μοντέλα Groq: llama-3.3-70b-versatile, llama-3.1-8b-instant, gemma2-9b-it
llm = LLM(
    model="groq/llama-3.3-70b-versatile",
    temperature=0.7  # Λίγη δημιουργικότητα, αλλά όχι υπερβολική
)

print("✅ LLM ορίστηκε:", llm.model)

✅ LLM ορίστηκε: groq/llama-3.3-70b-versatile


## Βήμα 2: Ορισμός των Πρακτόρων (Agents)

Κάθε `Agent` ορίζεται από τρία βασικά στοιχεία:

| Παράμετρος | Τι κάνει |
|---|---|
| `role` | Ο τίτλος/ρόλος του agent — επηρεάζει τον τρόπο που «σκέφτεται» |
| `goal` | Ο συγκεκριμένος στόχος που πρέπει να επιτύχει |
| `backstory` | Το «ιστορικό» του — δίνει πλαίσιο και προσωπικότητα |

> 💡 **Γιατί το backstory;** Τα LLMs αποδίδουν καλύτερα όταν έχουν πλαίσιο. Το backstory είναι essentially **prompt engineering** — πείθουμε το μοντέλο να «παίξει» έναν ρόλο.


In [2]:
from crewai import Agent

# ── Agent 1: Ερευνητής ──────────────────────────────────────────────────────
researcher = Agent(
    role="Senior Market Research Analyst",
    goal=(
        "Συλλέξε ολοκληρωμένες πληροφορίες για την εταιρεία {company}: "
        "κύρια προϊόντα/υπηρεσίες, θέση στην αγορά, πρόσφατες εξελίξεις, "
        "ανταγωνισμός και βασικά οικονομικά στοιχεία."
    ),
    backstory=(
        "Είσαι έμπειρος αναλυτής με 15 χρόνια στον κλάδο της τεχνολογίας. "
        "Ειδικεύεσαι στην ανάλυση εταιρειών υψηλής τεχνολογίας και έχεις εργαστεί "
        "σε κορυφαία investment banks. Η ανάλυσή σου είναι πάντα δομημένη, "
        "αντικειμενική και βασισμένη σε στοιχεία."
    ),
    llm=llm,
    verbose=True  # Εμφάνιση της σκέψης του agent σε πραγματικό χρόνο
)

# ── Agent 2: Αναλυτής/Σύμβουλος ─────────────────────────────────────────────
advisor = Agent(
    role="Investment Advisor",
    goal=(
        "Με βάση την έρευνα που έχει γίνει, γράψε μια επαγγελματική αναφορά "
        "επενδυτικής αξιολόγησης για την εταιρεία {company}. "
        "Η αναφορά πρέπει να περιλαμβάνει σαφή θέση (Αγορά / Αναμονή / Πώληση)."
    ),
    backstory=(
        "Είσαι senior investment advisor με εξειδίκευση στον τεχνολογικό τομέα. "
        "Γράφεις αναφορές για θεσμικούς επενδυτές και portfolio managers. "
        "Η γλώσσα σου είναι επαγγελματική αλλά κατανοητή. "
        "Πάντα παρουσιάζεις τα υπέρ και τα κατά πριν δώσεις σύσταση."
    ),
    llm=llm,
    verbose=True
)

print("✅ Δύο agents ορίστηκαν:")
print(f"  1. {researcher.role}")
print(f"  2. {advisor.role}")

✅ Δύο agents ορίστηκαν:
  1. Senior Market Research Analyst
  2. Investment Advisor


## Βήμα 3: Ορισμός Εργασιών (Tasks)

Κάθε `Task` αντιστοιχεί σε μια συγκεκριμένη εργασία που αναθέτουμε σε έναν agent.

Σημαντική διαφορά από ένα απλό prompt:
- Το `description` περιγράφει **τι πρέπει να γίνει**
- Το `expected_output` ορίζει **ακριβώς τι αναμένουμε** — λειτουργεί σαν acceptance criteria
- Το `context` μεταφέρει **το output της προηγούμενης task** στον επόμενο agent


In [3]:
from crewai import Task

# Η εταιρεία που θα αναλυθεί — αλλάξτε την για να δοκιμάσετε διαφορετικά!
COMPANY = "NVIDIA"

# ── Task 1: Έρευνα ───────────────────────────────────────────────────────────
research_task = Task(
    description=(
        f"Κάνε εκτενή ανάλυση της εταιρείας {COMPANY}. "
        "Κάλυψε τους παρακάτω τομείς:\n"
        "1. Επισκόπηση εταιρείας (ίδρυση, μέγεθος, τομέας)\n"
        "2. Κύρια προϊόντα και υπηρεσίες\n"
        "3. Ανταγωνιστικό πλεονέκτημα (moat)\n"
        "4. Κύριοι ανταγωνιστές\n"
        "5. Πρόσφατες εξελίξεις και τάσεις στον κλάδο"
    ),
    expected_output=(
        "Δομημένη αναφορά έρευνας σε 5 ενότητες με bullet points. "
        "Μέγιστο 400 λέξεις. Χρησιμοποίησε αγγλικούς όρους για τα τεχνικά/χρηματοοικονομικά."
    ),
    agent=researcher  # Αυτός ο agent είναι υπεύθυνος
)

# ── Task 2: Επενδυτική Αναφορά ───────────────────────────────────────────────
analysis_task = Task(
    description=(
        f"Με βάση την έρευνα για την {COMPANY}, "
        "ετοίμασε μια επενδυτική αναφορά για έναν portfolio manager. "
        "Δομή αναφοράς:\n"
        "- Executive Summary (2-3 προτάσεις)\n"
        "- Ισχυρά σημεία (Υπέρ επένδυσης)\n"
        "- Κίνδυνοι (Κατά επένδυσης)\n"
        "- Τελική Σύσταση: ΑΓΟΡΑ / ΑΝΑΜΟΝΗ / ΠΩΛΗΣΗ με αιτιολόγηση"
    ),
    expected_output=(
        "Επαγγελματική επενδυτική αναφορά σε 4 ενότητες. "
        "Η τελική σύσταση να είναι ξεκάθαρη και αιτιολογημένη."
    ),
    agent=advisor,
    context=[research_task]  # Ο advisor βλέπει το output του researcher
)

print(f"✅ Δύο tasks ορίστηκαν για: {COMPANY}")
print(f"  Task 1 → {researcher.role}")
print(f"  Task 2 → {advisor.role} (με context από Task 1)")

✅ Δύο tasks ορίστηκαν για: NVIDIA
  Task 1 → Senior Market Research Analyst
  Task 2 → Investment Advisor (με context από Task 1)


## Βήμα 4: Δημιουργία της Crew και Εκτέλεση

Η `Crew` είναι ο **ορχηστράτορας** — συντονίζει τους agents και τα tasks.

Υπάρχουν δύο τρόποι εκτέλεσης (`Process`):
- **`sequential`**: Tasks εκτελούνται σειριακά (ένα μετά το άλλο) — απλό και προβλέψιμο
- **`hierarchical`**: Ένας manager agent κατανέμει εργασίες δυναμικά — πιο ευέλικτο

Εδώ χρησιμοποιούμε `sequential` για να δούμε ξεκάθαρα τη ροή: **Researcher → Advisor**.


In [6]:
from crewai import Crew, Process

# Δημιουργία της Crew
investment_crew = Crew(
    agents=[researcher, advisor],
    tasks=[research_task, analysis_task],
    process=Process.sequential,  # Researcher πρώτα, μετά Advisor
    verbose=True
)

print("✅ Crew δημιουργήθηκε.")
print(f"   Agents: {len(investment_crew.agents)}")
print(f"   Tasks:  {len(investment_crew.tasks)}")
print(f"   Process: Sequential")
print("\n" + "="*60)
print("🚀 Εκκίνηση... (αναμένετε 30-60 δευτερόλεπτα)")
print("="*60 + "\n")

# Εκτέλεση — kickoff() ξεκινά τη ροή
result = investment_crew.kickoff(inputs={"company": COMPANY})

✅ Crew δημιουργήθηκε.
   Agents: 2
   Tasks:  2
   Process: Sequential

🚀 Εκκίνηση... (αναμένετε 30-60 δευτερόλεπτα)



╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 413c33bb-51d9-43c2-8153-f104b94db622                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Κάνε εκτενή ανάλυση της εταιρείας NVIDIA. Κάλυψε τους παρακάτω τομείς:                                   │
│  1. Επισκόπηση εταιρείας (ίδρυση, μέγεθος, τομέας)                                                              │
│  2. Κύρια προϊόντα και υπηρεσίες                                                                                │
│  3. Ανταγωνιστικό πλεονέκτημα (moat)                                                                            │
│  4. Κύριοι ανταγωνιστές                                                                                         │
│  5. Πρόσφατες εξελίξεις και τάσεις στον κλάδο                                                                   │
│  ID: abfc0856-79fa-40de-9bad-f0fefc6c5295                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Market Research Analyst                                                                          │
│                                                                                                                 │
│  Task: Κάνε εκτενή ανάλυση της εταιρείας NVIDIA. Κάλυψε τους παρακάτω τομείς:                                   │
│  1. Επισκόπηση εταιρείας (ίδρυση, μέγεθος, τομέας)                                                              │
│  2. Κύρια προϊόντα και υπηρεσίες                                                                                │
│  3. Ανταγωνιστικό πλεονέκτημα (moat)                                                                            │
│  4. Κύριοι ανταγωνιστές                                                                                         │
│  5. Πρόσφατες εξελίξεις και τάσεις στον κλάδο                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Market Research Analyst                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Επισκόπηση Εταιρείας**                                                                                       │
│  * Η NVIDIA ιδρύθηκε το 1993                                                                                    │
│  * Είναι μια εταιρεία που δραστηριοποιείται στον τομέα της υψηλής τεχνολογίας, ειδικότερα στις γραφικές         │
│  καρτές, τις επεξεργαστικές μονάδες (GPU) και τις τεχνολογίες τεχνητής νοημοσύνης (AI)                          │
│  * Η NVIDIA είναι μια από τις μεγαλύτερες εταιρείες στον τομέα της τεχνολογίας, με μέγεθος που φτάνει τα 500    │
│  δισεκατομμύρια δολάρια                                                                                         │
│                                                                                                                 │
│  **Κύρια Προϊόντα και Υπηρεσίες**                                                                               │
│  * Γραφικές κάρτες (GeForce)                                                                                    │
│  * Επεξεργαστικές μονάδες (GPU) για επαγγελματική χρήση (Quadro)                                                │
│  * Τεχνολογίες τεχνητής νοημοσύνης (AI) και μηχανικής μάθησης (ML)                                              │
│  * Λογισμικό και υπηρεσίες για την ανάπτυξη εφαρμογών (SDKs) και την διαχείριση δεδομένων (Data Center)         │
│                                                                                                                 │
│  **Ανταγωνιστικό Πλεονέκτημα (Moat)**                                                                           │
│  * Η NVIDIA έχει ένα ανταγωνιστικό πλεονέκτημα λόγω της μεγάλης της εμπειρίας και της τεχνολογικής της ηγεσίας  │
│  στον τομέα των GPU και της AI                                                                                  │
│  * Η εταιρεία έχει μια στενή συνεργασία με τους μεγαλύτερους κατασκευαστές συστημάτων και τις εταιρείες         │
│  λογισμικού, مما της δίνει μια μοναδική θέση στην αγορά                                                         │
│                                                                                                                 │
│  **Κύριοι Ανταγωνιστές**                                                                                        │
│  * Advanced Micro Devices (AMD)                                                                                 │
│  * Intel Corporation                                                                                            │
│  * Google LLC (υπηρεσίες cloud και AI)                                                                          │
│  * Amazon Web Services (υπηρεσίες cloud και AI)                                                                 │
│                                                                                                                 │
│  **Πρόσφατες Εξελίξεις και Τάσεις**                                                                             │
│  * Η NVIDIA έχει επεκτείνει την παρουσία της στον τομέα της αυτονομίας οχημάτων με την ανάπτυξη της             │
│  τεχνολογίας Drive                                                                                              │
│  * Η εταιρεία έχει επίσης επενδύσει στην ανάπτυξη της τεχνολογίας 5G και της edge computing                     │
│  * Η NVIDIA έχει ανακοινώσει μια σειρά από συνεργασίες 

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investment Advisor                                                                                      │
│                                                                                                                 │
│  Task: Με βάση την έρευνα για την NVIDIA, ετοίμασε μια επενδυτική αναφορά για έναν portfolio manager. Δομή      │
│  αναφοράς:                                                                                                      │
│  - Executive Summary (2-3 προτάσεις)                                                                            │
│  - Ισχυρά σημεία (Υπέρ επένδυσης)                                                                               │
│  - Κίνδυνοι (Κατά επένδυσης)                                                                                    │
│  - Τελική Σύσταση: ΑΓΟΡΑ / ΑΝΑΜΟΝΗ / ΠΩΛΗΣΗ με αιτιολόγηση                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Investment Advisor                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Επαγγελματική Επενδυτική Αναφορά: NVIDIA**                                                                   │
│                                                                                                                 │
│  ### Executive Summary                                                                                          │
│  Η NVIDIA είναι μια ηγετική εταιρεία στον τομέα της υψηλής τεχνολογίας, με ιδιαίτερη εξειδίκευση στις γραφικές  │
│  κάρτες, τις επεξεργαστικές μονάδες (GPU) και τις τεχνολογίες τεχνητής νοημοσύνης (AI). Με ένα ανταγωνιστικό    │
│  πλεονέκτημα λόγω της μεγάλης της εμπειρίας και της τεχνολογικής της ηγεσίας, η NVIDIA βρίσκεται σε μια υψηλά   │
│  конкурριακή θέση στην αγορά. Η εταιρεία συνεχίζει να επεκτείνει την παρουσία της σε νέους τομείς, όπως η       │
│  αυτονομία οχημάτων και η edge computing, και έχει ανακοινώσει σημαντικές συνεργασίες με μεγάλες εταιρείες.     │
│                                                                                                                 │
│  ### Ισχυρά Σημεία (Υπέρ Επένδυσης)                                                                             │
│  - **Ηγετική Θέση στην Αγορά**: Η NVIDIA είναι ηγέτης στον τομέα των GPU και της AI, με μια ισχυρή παρουσία     │
│  στην αγορά.                                                                                                    │
│  - **Ανταγωνιστικό Πλεονέκτημα**: Η εταιρεία έχει ένα ανταγωνιστικό πλεονέκτημα λόγω της μεγάλης της εμπειρίας  │
│  και της τεχνολογικής της ηγεσίας, το οποίο της δίνει μια μοναδική θέση στην αγορά.                             │
│  - **Επεκταμένη Παρουσία**: Η NVIDIA έχει επεκτείνει την παρουσία της σε νέους τομείς, όπως η αυτονομία         │
│  οχημάτων, η 5G και η edge computing, προσφέροντας νέες ευκαιρίες για ανάπτυξη.                                 │
│  - **Συnergικές Συνεργασίες**: Η εταιρεία έχει ανακοινώσει σημαντικές συνεργασίες με μεγάλες εταιρείες, όπως η  │
│  Google και η Microsoft, για την ανάπτυξη υπηρεσιών cloud και AI.                                               │
│  - **Ισχυρό Ισοζύγιο**: Η NVIDIA έχει ένα ισχυρό ισοζύγιο, με σημαντικά κεφάλαια και μια σταθερή ροή εσόδων.    │
│                                                                                                                 │
│  ### Κίνδυνοι (Κατά Επένδυσης)                                                                                  │
│  - **Ανταγωνισμός**: Η NVIDIA αντιμετωπίζει σημαντικό ανταγωνισμό από εταιρείες όπως η AMD, η Intel και η       │
│  Google, οι οποίες προσπαθούν να μειώσουν την ηγετική της θέση στην αγορά.                                      │
│  - **Τεχνολογικές Αλλαγές**: Οι ταχείς τεχνολογικές αλλαγές στον τομέα της υψηλής τεχνολογίας possono να        │
│  επηρεάσουν την ανταγωνιστικότητα της εταιρείας.                                                                │
│  - **Εξάρτηση από Μερικά Προϊόντα**: Η NVIDIA εξαρτάται σημαντικά από μερικά προϊόντα, όπως τις γραφικές        │
│  κάρτες GeForce, και μια πτώση στην πώληση αυτών των προϊόντων μπορεί να επηρεάσει αρνητικά την εταιρεία.       │
│  - **Επενδύσεις σε Νέους Τομείς**: Οι επενδύσεις της εταιρείας σε νέους τομείς, όπως η αυτονομία οχημάτων και   │
│  η edge computing, peuvent να μην αποδώσουν τα αναμενόμενα αποτελέσματα.                                        │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Με βάση την έρευνα για την NVIDIA, ετοίμασε μια επενδυτική αναφορά για έναν portfolio manager. Δομή      │
│  αναφοράς:                                                                                                      │
│  - Executive Summary (2-3 προτάσεις)                                                                            │
│  - Ισχυρά σημεία (Υπέρ επένδυσης)                                                                               │
│  - Κίνδυνοι (Κατά επένδυσης)                                                                                    │
│  - Τελική Σύσταση: ΑΓΟΡΑ / ΑΝΑΜΟΝΗ / ΠΩΛΗΣΗ με αιτιολόγηση                                                      │
│  Agent: Investment Advisor                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 413c33bb-51d9-43c2-8153-f104b94db622                                                                       │
│  Final Output: **Επαγγελματική Επενδυτική Αναφορά: NVIDIA**                                                     │
│                                                                                                                 │
│  ### Executive Summary                                                                                          │
│  Η NVIDIA είναι μια ηγετική εταιρεία στον τομέα της υψηλής τεχνολογίας, με ιδιαίτερη εξειδίκευση στις γραφικές  │
│  κάρτες, τις επεξεργαστικές μονάδες (GPU) και τις τεχνολογίες τεχνητής νοημοσύνης (AI). Με ένα ανταγωνιστικό    │
│  πλεονέκτημα λόγω της μεγάλης της εμπειρίας και της τεχνολογικής της ηγεσίας, η NVIDIA βρίσκεται σε μια υψηλά   │
│  конкурριακή θέση στην αγορά. Η εταιρεία συνεχίζει να επεκτείνει την παρουσία της σε νέους τομείς, όπως η       │
│  αυτονομία οχημάτων και η edge computing, και έχει ανακοινώσει σημαντικές συνεργασίες με μεγάλες εταιρείες.     │
│                                                                                                                 │
│  ### Ισχυρά Σημεία (Υπέρ Επένδυσης)                                                                             │
│  - **Ηγετική Θέση στην Αγορά**: Η NVIDIA είναι ηγέτης στον τομέα των GPU και της AI, με μια ισχυρή παρουσία     │
│  στην αγορά.                                                                                                    │
│  - **Ανταγωνιστικό Πλεονέκτημα**: Η εταιρεία έχει ένα ανταγωνιστικό πλεονέκτημα λόγω της μεγάλης της εμπειρίας  │
│  και της τεχνολογικής της ηγεσίας, το οποίο της δίνει μια μοναδική θέση στην αγορά.                             │
│  - **Επεκταμένη Παρουσία**: Η NVIDIA έχει επεκτείνει την παρουσία της σε νέους τομείς, όπως η αυτονομία         │
│  οχημάτων, η 5G και η edge computing, προσφέροντας νέες ευκαιρίες για ανάπτυξη.                                 │
│  - **Συnergικές Συνεργασίες**: Η εταιρεία έχει ανακοινώσει σημαντικές συνεργασίες με μεγάλες εταιρείες, όπως η  │
│  Google και η Microsoft, για την ανάπτυξη υπηρεσιών cloud και AI.                                               │
│  - **Ισχυρό Ισοζύγιο**: Η NVIDIA έχει ένα ισχυρό ισοζύγιο, με σημαντικά κεφάλαια και μια σταθερή ροή εσόδων.    │
│                                                                                                                 │
│  ### Κίνδυνοι (Κατά Επένδυσης)                                                                                  │
│  - **Ανταγωνισμός**: Η NVIDIA αντιμετωπίζει σημαντικό ανταγωνισμό από εταιρείες όπως η AMD, η Intel και η       │
│  Google, οι οποίες προσπαθούν να μειώσουν την ηγετική της θέση στην αγορά.                                      │
│  - **Τεχνολογικές Αλλαγές**: Οι ταχείς τεχνολογικές αλλαγές στον τομέα της υψηλής τεχνολογίας possono να        │
│  επηρεάσουν την ανταγωνιστικότητα της εταιρείας.                                                                │
│  - **Εξάρτηση από Μερικά Προϊόντα**: Η NVIDIA εξαρτάται σημαντικά από μερικά προϊόντα, όπως τις γραφικές        │
│  κάρτες GeForce, και μια πτώση στην πώληση αυτών των προϊόντων μπορεί να επηρεάσει αρνητικά την εταιρεία.       │
│  - **Επενδύσεις σε Νέους Τομείς**: Οι επενδύσεις της εταιρείας σε νέους τομείς, όπως η αυτονομία οχημάτων και   │
│  η edge computing, peuvent να μην αποδώσουν τα αναμενόμενα αποτελέσματα.                                        │
│                                                       

## Αποτελέσματα

Μετά την εκτέλεση της Crew, το αντικείμενο `result` περιέχει την έξοδο όλης της αλυσίδας:

| Ιδιότητα | Περιεχόμενο |
|---|---|
| `result.raw` | Η τελική έξοδος ως απλό κείμενο (string) |
| `result.tasks_output` | Λίστα με τα outputs κάθε task ξεχωριστά |
| `result.token_usage` | Πόσα tokens χρησιμοποιήθηκαν συνολικά |

Εδώ εκτυπώνουμε το `result.raw` — δηλαδή την τελική επενδυτική αναφορά που έγραψε ο Advisor.

In [7]:
# Εμφάνιση τελικού αποτελέσματος
print("\n" + "="*60)
print(f"📊 ΤΕΛΙΚΗ ΑΝΑΦΟΡΑ: {COMPANY}")
print("="*60)
print(result.raw)


📊 ΤΕΛΙΚΗ ΑΝΑΦΟΡΑ: NVIDIA
**Επαγγελματική Επενδυτική Αναφορά: NVIDIA**

### Executive Summary
Η NVIDIA είναι μια ηγετική εταιρεία στον τομέα της υψηλής τεχνολογίας, με ιδιαίτερη εξειδίκευση στις γραφικές κάρτες, τις επεξεργαστικές μονάδες (GPU) και τις τεχνολογίες τεχνητής νοημοσύνης (AI). Με ένα ανταγωνιστικό πλεονέκτημα λόγω της μεγάλης της εμπειρίας και της τεχνολογικής της ηγεσίας, η NVIDIA βρίσκεται σε μια υψηλά конкурριακή θέση στην αγορά. Η εταιρεία συνεχίζει να επεκτείνει την παρουσία της σε νέους τομείς, όπως η αυτονομία οχημάτων και η edge computing, και έχει ανακοινώσει σημαντικές συνεργασίες με μεγάλες εταιρείες.

### Ισχυρά Σημεία (Υπέρ Επένδυσης)
- **Ηγετική Θέση στην Αγορά**: Η NVIDIA είναι ηγέτης στον τομέα των GPU και της AI, με μια ισχυρή παρουσία στην αγορά.
- **Ανταγωνιστικό Πλεονέκτημα**: Η εταιρεία έχει ένα ανταγωνιστικό πλεονέκτημα λόγω της μεγάλης της εμπειρίας και της τεχνολογικής της ηγεσίας, το οποίο της δίνει μια μοναδική θέση στην αγορά.
- **Επεκταμένη Παρο

## Παρατηρήσεις & Ερωτήσεις για Συζήτηση

Αφού εκτελέσετε το notebook, σκεφτείτε τα παρακάτω:

**1. Παρατήρηση στο verbose output:**
Τι είδατε στην έξοδο ενώ έτρεχαν οι agents; Πώς «σκεφτόταν» ο καθένας;

**2. Σύνδεση με τη θεωρία:**
- Ποια είναι τα **Beliefs** του Researcher; (= η γνώση του LLM για την εταιρεία)
- Ποιο είναι το **Goal/Intention** του Advisor;
- Ποιο είναι το «εργαλείο» (Tool) που χρησιμοποιεί ο Advisor; (= το context από τον Researcher)

**3. Αρχιτεκτονική:**
Αυτό είναι **Sequential** process. Σε ποια περίπτωση θα χρησιμοποιούσατε Hierarchical;

**4. Περιορισμοί:**
Τι αδυναμία βλέπετε στο σύστημα αυτό; (Hint: τα δεδομένα είναι πρόσφατα;)


## ✏️ Άσκηση: Δοκιμάστε Διαφορετικές Εταιρείες & Ρόλους

Τροποποιήστε τον παρακάτω κώδικα για να:
1. Αναλύσετε μια ελληνική εταιρεία (π.χ. OTE, Jumbo, ΔΕΗ)
2. Προσθέσετε έναν τρίτο agent: **Risk Officer** που αξιολογεί τους κινδύνους
3. Αλλάξετε τη `temperature` και παρατηρήστε αν η αναφορά αλλάζει


In [8]:
# ✏️ Βήμα Α: Αλλάξτε εδώ την εταιρεία που θέλετε να αναλύσετε
MY_COMPANY = "Apple"  # ← Αλλάξτε εδώ (π.χ. "OTE", "Jumbo", "Tesla", "ΔΕΗ")

print(f"🏢 Εταιρεία για ανάλυση: {MY_COMPANY}")

🚀 Εκτέλεση με 3 agents για: Apple


### Agent 3: Chief Risk Officer

Προσθέτουμε έναν **τρίτο agent** στην ομάδα — τον υπεύθυνο ανάλυσης κινδύνου.

Το `Agent` είναι το βασικό δομικό στοιχείο του CrewAI. Κάθε agent:
- Έχει έναν **`role`** (τίτλος) που καθορίζει τον τρόπο «σκέψης» του
- Έχει ένα **`goal`** (στόχος) — τι πρέπει να επιτύχει σε αυτή την εκτέλεση
- Έχει ένα **`backstory`** (βιογραφικό) — prompt engineering που δίνει πλαίσιο και προσωπικότητα στο LLM
- Χρησιμοποιεί ένα **`llm`** (γλωσσικό μοντέλο) ως «εγκέφαλο»
- Με **`verbose=True`** εκτυπώνει τη σκέψη του σε πραγματικό χρόνο

In [ ]:
from crewai import Agent

# Ορισμός του τρίτου agent: Chief Risk Officer
risk_officer = Agent(
    role="Chief Risk Officer",
    goal=f"Εντόπισε και αξιολόγησε τους κύριους κινδύνους επένδυσης στην {MY_COMPANY}.",
    backstory=(
        "Ειδικεύεσαι στην ανάλυση ρίσκου επενδύσεων. "
        "Εντοπίζεις κινδύνους που άλλοι παραβλέπουν: "
        "ρυθμιστικούς, τεχνολογικούς, ανταγωνιστικούς και μακροοικονομικούς."
    ),
    llm=llm,
    verbose=True
)

print(f"✅ Agent ορίστηκε: {risk_officer.role}")

### Task 3: Ανάλυση Κινδύνων (Risk Task)

Η `Task` είναι η **εντολή εργασίας** που αναθέτουμε σε έναν agent. Τρία βασικά πεδία:

- **`description`**: Το αναλυτικό prompt — τι ακριβώς πρέπει να κάνει ο agent
- **`expected_output`**: Η μορφή του αναμενόμενου αποτελέσματος — λειτουργεί σαν *acceptance criteria*
- **`context`**: Λίστα από προηγούμενα tasks — ο agent διαβάζει τα outputs τους πριν ξεκινήσει

> 💡 Χρησιμοποιούμε `context=[research_task]` ώστε ο Risk Officer να έχει πρόσβαση στην έρευνα του Researcher και να μη χρειαστεί να επαναλάβει τη δουλειά από την αρχή.

In [ ]:
from crewai import Task

# Ορισμός της τρίτης task: Risk Matrix
risk_task = Task(
    description=(
        f"Με βάση την ανάλυση για την {MY_COMPANY}, "
        "κατάρτισε έναν πίνακα κινδύνων (Risk Matrix) με τους top-5 κινδύνους. "
        "Για κάθε κίνδυνο: Πιθανότητα (Υψηλή/Μέτρια/Χαμηλή) και Επίπτωση (Υψηλή/Μέτρια/Χαμηλή)."
    ),
    expected_output="Πίνακας 5 κινδύνων με πιθανότητα και επίπτωση.",
    agent=risk_officer,
    context=[research_task]  # Χρησιμοποιεί την ίδια έρευνα που έκανε ο Researcher
)

print(f"✅ Task ορίστηκε: Risk Matrix για {MY_COMPANY}")

### Extended Crew: 3 Agents, Sequential Process

Η `Crew` είναι ο **ορχηστράτορας** που συντονίζει agents και tasks. Βασικές παράμετροι:

| Παράμετρος | Τι κάνει |
|---|---|
| `agents` | Λίστα με όλους τους agents της ομάδας |
| `tasks` | Λίστα με τα tasks **με τη σειρά εκτέλεσης** (sequential) |
| `process` | `Process.sequential` = κάθε task εκτελείται αφού τελειώσει το προηγούμενο |
| `verbose` | Εκτυπώνει την πρόοδο κάθε βήματος |

Η ροή εδώ: **Researcher** → **Advisor** → **Risk Officer**

> ✏️ Αποσχολιάστε το `extended_crew.kickoff(...)` στο τέλος για να τρέξετε την ομάδα.

In [ ]:
from crewai import Crew, Process

# Νέα Crew με τρεις agents — η ροή είναι: Researcher → Advisor → Risk Officer
extended_crew = Crew(
    agents=[researcher, advisor, risk_officer],
    tasks=[research_task, analysis_task, risk_task],
    process=Process.sequential,  # Σειριακή εκτέλεση: κάθε agent περιμένει τον προηγούμενο
    verbose=True
)

print(f"✅ Extended Crew με {len(extended_crew.agents)} agents για: {MY_COMPANY}")
print(f"   Ροή: {' → '.join(a.role.split()[0] for a in extended_crew.agents)}")
print()
print("▶️  Για να εκτελέσετε, αποσχολιάστε τις επόμενες γραμμές:")

# Αποσχολιάστε για εκτέλεση:
# result2 = extended_crew.kickoff(inputs={"company": MY_COMPANY})
# print(result2.raw)

---

## Σύνοψη

| Έννοια | Υλοποίηση στο CrewAI |
|---|---|
| **Agent** = αυτόνομη οντότητα με στόχο | `Agent(role, goal, backstory, llm)` |
| **Task** = συγκεκριμένη ανάθεση | `Task(description, expected_output, agent)` |
| **Context** = επικοινωνία μεταξύ agents | `Task(context=[άλλο_task])` |
| **Crew** = ορχηστράτορας ομάδας | `Crew(agents, tasks, process)` |
| **Sequential Process** = αλυσίδα | Agent A → Agent B → Agent C |

### Περαιτέρω Εξερεύνηση

- **Groq models**: `llama-3.1-8b-instant` (πιο γρήγορο), `gemma2-9b-it` (Google)
- **Web search**: Προσθέστε `SerperDevTool` ή `DuckDuckGoSearchRun` για πραγματική αναζήτηση
- **Hierarchical process**: Ένας manager agent κατανέμει εργασίες αυτόματα
- **Memory**: Οι agents μπορούν να «θυμούνται» προηγούμενες συνεδρίες
- **Docs**: [docs.crewai.com](https://docs.crewai.com)
